# Module Analysis

Exploratory analysis of PPI module outputs.

In [1]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", "{:.3f}".format)

ROOT = "../outputs/network/modules"

### Table 1 — Overall Graph & Module Quality

Key graph and Leiden statistics for every (method, strategy) combination. With this setting. Seem the top10Node strategy generate a more reasonable graph.

In [2]:
overview = pd.read_csv(f"{ROOT}/module_detection_overview.csv")

graph_cols = [
    "method", "strategy",
    "n_nodes", "n_edges",
    "n_modules", "n_modules_ge_min", "n_singletons",
    "module_size_min", "module_size_median", "module_size_mean", "module_size_max",
    "n_mcode_cores",
    "top_module_size", "top_mcode_size",
]

t1 = (
    overview[graph_cols]
    .sort_values(["method", "strategy"])
    .reset_index(drop=True)
)

def flag(row):
    notes = []
    if row["module_size_max"] > 900:
        notes.append("⚠ giant module")
    if row["module_size_median"] <= 2 and row["n_modules"] > 100:
        notes.append("⚠ fragmented")
    if row["top_mcode_size"] > 150:
        notes.append("⚠ MCODE core too large")
    if row["n_nodes"] < 1500:
        notes.append("⚠ low coverage")
    return " | ".join(notes) if notes else "✓"

t1["notes"] = t1.apply(flag, axis=1)
display(t1)

,method,strategy,n_nodes,n_edges,n_modules,n_modules_ge_min,n_singletons,module_size_min,module_size_median,module_size_mean,module_size_max,n_mcode_cores,top_module_size,top_mcode_size,notes
0,autoencoder_cosine,threshold_0p85,7220,62933,385,142,0,2,2.000,18.753,436,1051,436,99,⚠ fragmented
1,autoencoder_cosine,top100000,8871,100000,278,110,0,2,2.000,31.910,508,1393,508,106,⚠ fragmented
2,autoencoder_cosine,top10perNodes,12142,93250,47,47,0,30,235.000,258.340,664,976,664,15,✓
3,combined_pearson_jaccard,threshold_0p85,4956,100000,181,73,1,1,2.000,27.381,453,1235,453,149,⚠ fragmented
4,combined_pearson_jaccard,top100000,4956,100000,184,73,0,2,2.000,26.935,526,1235,526,149,⚠ fragmented
5,combined_pearson_jaccard,top10perNodes,11356,101991,39,39,0,46,315.000,291.179,780,935,780,23,✓
6,combined_spearman_jaccard,threshold_0p85,4822,100000,202,81,9,1,2.000,23.871,543,1153,543,185,⚠ fragmented | ⚠ MCODE core too large
7,combined_spearman_jaccard,top100000,4822,100000,214,81,15,1,2.000,22.533,750,1153,750,185,⚠ fragmented | ⚠ MCODE core too large
8,combined_spearman_jaccard,top10perNodes,11356,101788,41,41,0,8,279.000,276.976,746,839,746,18,✓
9,composite_rbf_jaccard,threshold_0p85,2492,100000,190,86,30,1,2.000,13.116,351,746,351,249,⚠ fragmented | ⚠ MCODE core too large


### Table 2 — Top Leiden Modules Overview

Top 10 Leiden modules per (method, strategy): size, density, mean/median edge weight, key proteins.

In [3]:
top_modules = pd.read_csv(f"{ROOT}/top_modules_overview.csv")

leiden_cols = [
    "method", "strategy", "module_id",
    "size", "n_edges", "density",
    "mean_weight", "median_weight", "max_weight",
    "mean_degree", "max_degree",
    "key_proteins",
]

t2 = (
    top_modules[leiden_cols]
    .sort_values(["method", "strategy", "size"], ascending=[True, True, False])
    .reset_index(drop=True)
)

# Rank within each (method, strategy) group
t2["rank"] = (
    t2.groupby(["method", "strategy"])["size"]
    .rank(ascending=False, method="first")
    .astype(int)
)

display(t2[["method", "strategy", "rank"] + leiden_cols[2:]])

,method,strategy,rank,module_id,size,n_edges,density,mean_weight,median_weight,max_weight,mean_degree,max_degree,key_proteins
0,autoencoder_cosine,threshold_0p85,1,L0001,436,2186,0.023,0.876,0.870,0.995,10.028,59,EFHD2;ACTR2;FGD4;MAP3K3;COTL1;MAPK3;CSK;ACTR3;ARHGAP30;GRB2
1,autoencoder_cosine,threshold_0p85,2,L0002,347,3902,0.065,0.881,0.875,0.989,22.490,113,TLR8;LRRC25;SIGLEC9;IL1RN;NCF2;RBM47;HVCN1;MEFV;FBP1;FGR
2,autoencoder_cosine,threshold_0p85,3,L0003,334,1258,0.023,0.875,0.869,0.989,7.533,36,POGZ;EHMT1;ZNF574;CHD4;SMARCE1;EIF4A3;POLR3A;GTF3C4;EHMT...
3,autoencoder_cosine,threshold_0p85,4,L0004,313,1434,0.029,0.881,0.875,0.988,9.163,49,EDEM2;DDOST;RPN2;TMED10;MLEC;FKBP2;RPN1;STT3A;NCLN;GANAB
4,autoencoder_cosine,threshold_0p85,5,L0005,310,1962,0.041,0.878,0.873,0.968,12.658,74,NELFA;NELFE;PHF3;SNW1;ZC3H4;RBBP6;SART1;RSRC1;RBM33;CWC15
...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,transformer_cosine,top10perNodes,6,L0006,449,2154,0.021,0.910,0.911,0.990,9.595,23,RAB44;COMMD6;TXNDC17;AP5B1;CRACR2A;VPS29;KIAA0556;CCDC22...
296,transformer_cosine,top10perNodes,7,L0007,441,2124,0.022,0.913,0.914,0.987,9.633,27,PHF20;ZBED1;POU2F1;PAXIP1;RBM15B;RBBP5;ZNHIT1;DEK;TBC1D3...
297,transformer_cosine,top10perNodes,8,L0008,429,2093,0.023,0.911,0.912,0.988,9.758,28,FOXC1;DSC2;CERS1;TRIM2;ZNF711;ABTB2;RHPN1;CUL7;MTCL1;NDN
298,transformer_cosine,top10perNodes,9,L0009,428,2548,0.028,0.927,0.929,0.993,11.907,33,TIMM9;GTPBP6;AGK;POLG2;CYC1;PHB2;FUNDC1;COQ9;AK3;DLST


### Table 3 — Top MCODE Cores Overview

Top 10 MCODE dense cores per (method, strategy): size, density, MCODE score, mean/median weight, key proteins.

In [4]:
top_cores = pd.read_csv(f"{ROOT}/top_mcode_cores_overview.csv")

mcode_cols = [
    "method", "strategy", "core_id",
    "size", "n_edges", "density", "mcode_score",
    "mean_weight", "median_weight", "max_weight",
    "seed_protein", "key_proteins",
]

t3 = (
    top_cores[mcode_cols]
    .sort_values(["method", "strategy", "mcode_score"], ascending=[True, True, False])
    .reset_index(drop=True)
)

t3["rank"] = (
    t3.groupby(["method", "strategy"])["mcode_score"]
    .rank(ascending=False, method="first")
    .astype(int)
)

display(t3[["method", "strategy", "rank"] + mcode_cols[2:]])

,method,strategy,rank,core_id,size,n_edges,density,mcode_score,mean_weight,median_weight,max_weight,seed_protein,key_proteins
0,autoencoder_cosine,threshold_0p85,1,MC0001,99,4642,0.957,94.735,0.927,0.930,0.992,CCNA2,KIF22;CKAP2L;KIF11;HMMR;SGO2;CENPF;TTK;KIF2C;TOP2A;CCNB1
1,autoencoder_cosine,threshold_0p85,2,MC0002,92,4082,0.975,89.714,0.927,0.930,0.992,CDC45,KIF22;KIF11;CKAP2L;CENPF;SGO2;TOP2A;TTK;SKA1;KIF20B;HMMR
2,autoencoder_cosine,threshold_0p85,3,MC0003,84,3428,0.983,82.602,0.933,0.937,0.992,MCM10,KIF22;HMMR;CKAP2L;KIF11;INCENP;SGO2;TOP2A;TTK;CENPF;RACGAP1
3,autoencoder_cosine,threshold_0p85,4,MC0004,77,2827,0.966,74.395,0.929,0.933,0.992,SPC24,KIF2C;HMMR;KIF22;CKAP2L;TTK;DIAPH3;KIF11;KNL1;SGO2;CCNB1
4,autoencoder_cosine,threshold_0p85,5,MC0005,81,2962,0.914,74.050,0.920,0.916,0.997,TSPAN9,PARVB;MMRN1;ITGA2B;ITGB3;THBS1;KALRN;GP9;TUBB1;GNAZ;GP1BB
...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,transformer_cosine,top10perNodes,6,MC0006,16,75,0.625,10.000,0.961,0.963,0.991,RPLP0,RPLP0;RPS3A;RPL38;RPS19;RPS24;RPS25;RPL34;RPS8;RPL31;RPLP2
296,transformer_cosine,top10perNodes,7,MC0007,11,50,0.909,10.000,0.983,0.982,0.997,ASB2,LFNG;FFAR2;UBE2W;ASB2;IL1RL1;CCL23;FNDC8;MYL7;OLIG1;AMN
297,transformer_cosine,top10perNodes,8,MC0008,10,42,0.933,9.333,0.976,0.976,0.994,WDR36,TBL3;HEATR1;WDR36;NOL11;UTP6;WDR75;UTP4;UTP15;WDR3;URB1
298,transformer_cosine,top10perNodes,9,MC0009,15,65,0.619,9.286,0.927,0.927,0.983,FBXO7,FBXO7;UBAC1;BSDC1;RABGAP1L;TBCEL;MARCH8;EPB41;MPP1;CALCO...


### Table 4 — Per-Method Summary Statistics

Aggregated Leiden and MCODE statistics per method, collapsed across strategies.

In [5]:
t4 = (
    overview
    .groupby("method")
    .agg(
        n_strategies=("strategy", "count"),
        nodes_mean=("n_nodes", "mean"),
        nodes_max=("n_nodes", "max"),
        leiden_modules_mean=("n_modules", "mean"),
        leiden_ge_min_mean=("n_modules_ge_min", "mean"),
        singletons_mean=("n_singletons", "mean"),
        median_module_size_mean=("module_size_median", "mean"),
        max_module_size_max=("module_size_max", "max"),
        mcode_cores_mean=("n_mcode_cores", "mean"),
        top_mcode_size_mean=("top_mcode_size", "mean"),
        top_mcode_size_max=("top_mcode_size", "max"),
    )
    .round(1)
    .reset_index()
    .sort_values("top_mcode_size_mean")
)

display(t4)

,method,n_strategies,nodes_mean,nodes_max,leiden_modules_mean,leiden_ge_min_mean,singletons_mean,median_module_size_mean,max_module_size_max,mcode_cores_mean,top_mcode_size_mean,top_mcode_size_max
5,pearson,3,7835.700,11356,136.700,67.000,0.000,68.700,623,644.000,56.000,119
7,spearman,3,7655.300,11356,149.000,73.700,0.000,57.200,968,538.000,65.700,137
6,pearson_adjusted,3,5835.000,11356,142.000,65.300,0.300,100.700,837,780.300,69.300,150
0,autoencoder_cosine,3,9411.000,12142,236.700,99.700,0.000,79.700,664,1140.000,73.300,106
9,transformer_cosine,3,10330.000,12142,374.700,161.300,0.000,89.700,735,886.700,78.700,106
8,spearman_adjusted,3,5726.300,11356,146.700,67.700,5.000,91.300,776,694.300,81.300,177
1,combined_pearson_jaccard,3,7089.300,11356,134.700,61.700,0.300,106.300,780,1135.000,107.000,149
2,combined_spearman_jaccard,3,7000.000,11356,152.300,67.700,8.000,94.300,750,1048.300,129.300,185
4,intensity_rbf,3,5511.300,11356,143.700,70.700,18.000,87.300,785,810.300,166.000,237
3,composite_rbf_jaccard,3,5446.700,11356,138.700,68.700,20.300,91.500,1102,872.700,174.300,249


### Table 5 — Method × Strategy Comparison (Structural)

Qualitative comparison of each similarity method across graph strategies, derived from the evaluation.

In [6]:
# Compute per-method structural flags from the overview table (no hardcoding).
# Thresholds mirror the evaluation criteria discussed in the report.

GIANT_MODULE_THRESH = 900      # top module size flagged as giant
FRAG_MEDIAN_THRESH = 2        # median module size ≤ this → fragmented
LOW_COV_THRESH = 1500         # node count below this → low protein coverage
MCODE_LARGE_THRESH = 150      # top MCODE core size above this → too large

def classify_row(row):
    flags = []
    if row["module_size_max"] > GIANT_MODULE_THRESH:
        flags.append("giant_module")
    if row["module_size_median"] <= FRAG_MEDIAN_THRESH and row["n_modules"] > 100:
        flags.append("fragmented")
    if row["n_nodes"] < LOW_COV_THRESH:
        flags.append("low_coverage")
    if row["top_mcode_size"] > MCODE_LARGE_THRESH:
        flags.append("mcode_core_too_large")
    return "; ".join(flags) if flags else "ok"

ov = overview.copy()
ov["flags"] = ov.apply(classify_row, axis=1)

# Aggregate to method level
t5 = (
    ov.groupby("method")
    .agg(
        n_strategies=("strategy", "count"),
        # protein coverage
        nodes_min=("n_nodes", "min"),
        nodes_max=("n_nodes", "max"),
        # Leiden quality
        max_module_size_max=("module_size_max", "max"),
        median_size_mean=("module_size_median", "mean"),
        n_modules_mean=("n_modules", "mean"),
        n_ge_min_mean=("n_modules_ge_min", "mean"),
        singletons_mean=("n_singletons", "mean"),
        # MCODE quality
        top_mcode_size_min=("top_mcode_size", "min"),
        top_mcode_size_max=("top_mcode_size", "max"),
        top_mcode_size_mean=("top_mcode_size", "mean"),
        # Flags
        n_flagged_configs=("flags", lambda s: (s != "ok").sum()),
        flag_types=("flags", lambda s: " | ".join(sorted(set("; ".join(s).split("; ")) - {"ok", ""}))),
    )
    .round(1)
    .reset_index()
    .sort_values("top_mcode_size_mean")
)

# Recommended strategy: ok configs per method
ok_strategies = (
    ov[ov["flags"] == "ok"]
    .groupby("method")["strategy"]
    .apply(lambda x: "; ".join(sorted(x)))
    .rename("ok_strategies")
)
t5 = t5.merge(ok_strategies, on="method", how="left")
t5["ok_strategies"] = t5["ok_strategies"].fillna("none")

display(t5)

,method,n_strategies,nodes_min,nodes_max,max_module_size_max,median_size_mean,n_modules_mean,n_ge_min_mean,singletons_mean,top_mcode_size_min,top_mcode_size_max,top_mcode_size_mean,n_flagged_configs,flag_types,ok_strategies
0,pearson,3,2596,11356,623,68.700,136.700,67.000,0.000,14,119,56.000,1,fragmented,top100000; top10perNodes
1,spearman,3,1904,11356,968,57.200,149.000,73.700,0.000,18,137,65.700,2,fragmented | giant_module,top10perNodes
2,pearson_adjusted,3,1143,11356,837,100.700,142.000,65.300,0.300,23,150,69.300,2,fragmented | low_coverage,top10perNodes
3,autoencoder_cosine,3,7220,12142,664,79.700,236.700,99.700,0.000,15,106,73.300,2,fragmented,top10perNodes
4,transformer_cosine,3,9424,12142,735,89.700,374.700,161.300,0.000,24,106,78.700,2,fragmented,top10perNodes
5,spearman_adjusted,3,962,11356,776,91.300,146.700,67.700,5.000,26,177,81.300,2,fragmented | low_coverage | mcode_core_too_large,top10perNodes
6,combined_pearson_jaccard,3,4956,11356,780,106.300,134.700,61.700,0.300,23,149,107.000,2,fragmented,top10perNodes
7,combined_spearman_jaccard,3,4822,11356,750,94.300,152.300,67.700,8.000,18,185,129.300,2,fragmented | mcode_core_too_large,top10perNodes
8,intensity_rbf,3,2589,11356,785,87.300,143.700,70.700,18.000,24,237,166.000,2,fragmented | mcode_core_too_large,top10perNodes
9,composite_rbf_jaccard,3,2492,11356,1102,91.500,138.700,68.700,20.300,25,249,174.300,3,fragmented | giant_module | mcode_core_too_large,none


### Table 6 — Leiden–MCODE Consistency

Estimated dominant Leiden fraction for top MCODE cores per configuration (higher = MCODE core is well-nested in a single Leiden module).

In [7]:
import os

# For each (method, strategy) directory, load:
#   module_assignments.csv  → protein → leiden module_id
#   mcode_core_members.csv  → core_id, protein
#   mcode_cores.csv         → core_id, mcode_score, size, density
# Then compute the dominant Leiden fraction for the top-scoring MCODE core.

records = []

for method in sorted(os.listdir(ROOT)):
    method_path = os.path.join(ROOT, method)
    if not os.path.isdir(method_path):
        continue
    for strategy in sorted(os.listdir(method_path)):
        strategy_path = os.path.join(method_path, strategy)
        assign_f  = os.path.join(strategy_path, "module_assignments.csv")
        members_f = os.path.join(strategy_path, "mcode_core_members.csv")
        cores_f   = os.path.join(strategy_path, "mcode_cores.csv")
        if not all(os.path.exists(f) for f in [assign_f, members_f, cores_f]):
            continue

        assignments = pd.read_csv(assign_f)[["protein", "module_id"]]
        members     = pd.read_csv(members_f)[["core_id", "protein"]]
        cores       = pd.read_csv(cores_f)[["core_id", "mcode_score", "size", "density"]]

        # Top core by MCODE score
        top = cores.sort_values("mcode_score", ascending=False).iloc[0]
        top_core_id = top["core_id"]

        # Members of that core, merged with Leiden assignments
        core_members = members[members["core_id"] == top_core_id][["protein"]]
        merged = core_members.merge(assignments, on="protein", how="left")

        if len(merged) == 0:
            continue

        leiden_counts = merged["module_id"].value_counts()
        dominant_module    = leiden_counts.index[0]
        dominant_count     = int(leiden_counts.iloc[0])
        dominant_fraction  = dominant_count / len(merged)

        # How many distinct Leiden modules does the core span?
        n_leiden_modules = leiden_counts.shape[0]

        records.append({
            "method":               method,
            "strategy":             strategy,
            "top_core_id":          top_core_id,
            "top_core_score":       round(float(top["mcode_score"]), 1),
            "top_core_size":        int(top["size"]),
            "top_core_density":     round(float(top["density"]), 3),
            "dominant_leiden_mod":  dominant_module,
            "n_core_in_dom_module": dominant_count,
            "dominant_fraction":    round(dominant_fraction, 3),
            "n_leiden_modules_spanned": n_leiden_modules,
        })

t6 = (
    pd.DataFrame(records)
    .sort_values(["method", "strategy"])
    .reset_index(drop=True)
)

# Alignment label based on dominant fraction
def alignment_label(frac):
    if frac >= 0.90: return "Excellent"
    if frac >= 0.75: return "Good"
    if frac >= 0.55: return "Moderate"
    return "Poor"

t6["alignment"] = t6["dominant_fraction"].apply(alignment_label)

display(t6)

,method,strategy,top_core_id,top_core_score,top_core_size,top_core_density,dominant_leiden_mod,n_core_in_dom_module,dominant_fraction,n_leiden_modules_spanned,alignment
0,autoencoder_cosine,threshold_0p85,MC0001,94.700,99,0.957,L0010,99,1.000,1,Excellent
1,autoencoder_cosine,top100000,MC0001,102.600,106,0.968,L0010,106,1.000,1,Excellent
2,autoencoder_cosine,top10perNodes,MC0001,12.300,15,0.819,L0008,15,1.000,1,Excellent
3,combined_pearson_jaccard,threshold_0p85,MC0001,133.100,149,0.893,L0005,48,0.322,8,Poor
4,combined_pearson_jaccard,top100000,MC0001,133.100,149,0.893,L0004,54,0.362,7,Poor
5,combined_pearson_jaccard,top10perNodes,MC0001,13.500,23,0.585,L0022,23,1.000,1,Excellent
6,combined_spearman_jaccard,threshold_0p85,MC0001,158.300,185,0.856,L0008,29,0.157,32,Poor
7,combined_spearman_jaccard,top100000,MC0001,158.300,185,0.856,L0010,30,0.162,35,Poor
8,combined_spearman_jaccard,top10perNodes,MC0001,12.700,18,0.706,L0017,18,1.000,1,Excellent
9,composite_rbf_jaccard,threshold_0p85,MC0001,205.700,249,0.826,L0009,22,0.088,105,Poor


### Table 7 — Recurrent Biological Themes

Co-abundance clusters detected across ≥ 2 methods, with representative proteins and biological interpretation.

In [8]:
# Identify recurrently detected proteins by counting how many (method, strategy)
# pairs each protein appears in across the top-10 Leiden modules and MCODE cores.
# High recurrence = likely a robust co-abundance signal.

# --- Leiden key proteins ---
leiden_rows = []
for _, row in top_modules.iterrows():
    for p in str(row["key_proteins"]).split(";"):
        leiden_rows.append({
            "protein": p.strip(),
            "method": row["method"],
            "strategy": row["strategy"],
            "module_id": row["module_id"],
            "module_size": row["size"],
        })

leiden_protein_df = pd.DataFrame(leiden_rows)

leiden_recurrence = (
    leiden_protein_df
    .groupby("protein")
    .agg(
        n_configs=("method", lambda x: x.shape[0]),          # appearances across configs
        n_methods=("method", "nunique"),                      # distinct similarity methods
        methods=("method", lambda x: "; ".join(sorted(set(x)))),
        mean_module_size=("module_size", "mean"),
    )
    .reset_index()
    .sort_values("n_methods", ascending=False)
)

# --- MCODE key proteins ---
mcode_rows = []
for _, row in top_cores.iterrows():
    for p in str(row["key_proteins"]).split(";"):
        mcode_rows.append({
            "protein": p.strip(),
            "method": row["method"],
            "strategy": row["strategy"],
            "core_id": row["core_id"],
            "mcode_score": row["mcode_score"],
        })

mcode_protein_df = pd.DataFrame(mcode_rows)

mcode_recurrence = (
    mcode_protein_df
    .groupby("protein")
    .agg(
        n_configs=("method", lambda x: x.shape[0]),
        n_methods=("method", "nunique"),
        methods=("method", lambda x: "; ".join(sorted(set(x)))),
        mean_mcode_score=("mcode_score", "mean"),
    )
    .reset_index()
    .sort_values("n_methods", ascending=False)
)

print("=== Top 30 Leiden key proteins by recurrence across methods ===")
display(leiden_recurrence.head(30))

print("\n=== Top 30 MCODE key proteins by recurrence across methods ===")
display(mcode_recurrence.head(30))

=== Top 30 Leiden key proteins by recurrence across methods ===


,protein,n_configs,n_methods,methods,mean_module_size
608,PDHB,20,10,autoencoder_cosine; combined_pearson_jaccard; combined_s...,421.950
622,PHF3,17,9,autoencoder_cosine; combined_pearson_jaccard; combined_s...,334.706
700,RBBP6,17,9,autoencoder_cosine; combined_pearson_jaccard; combined_s...,303.471
154,CHD4,18,9,autoencoder_cosine; combined_pearson_jaccard; combined_s...,329.056
304,FGR,19,9,autoencoder_cosine; combined_pearson_jaccard; combined_s...,461.000
388,IK,18,9,autoencoder_cosine; combined_pearson_jaccard; combined_s...,331.556
263,EHMT1,19,9,autoencoder_cosine; combined_pearson_jaccard; combined_s...,335.368
783,SF3B2,23,9,autoencoder_cosine; combined_pearson_jaccard; combined_s...,298.652
819,SNW1,13,8,autoencoder_cosine; combined_pearson_jaccard; combined_s...,353.385
526,NADK,17,8,combined_pearson_jaccard; combined_spearman_jaccard; com...,458.529



=== Top 30 MCODE key proteins by recurrence across methods ===


,protein,n_configs,n_methods,methods,mean_mcode_score
469,UTP4,13,10,autoencoder_cosine; combined_pearson_jaccard; combined_s...,12.092
433,TBL3,16,10,autoencoder_cosine; combined_pearson_jaccard; combined_s...,11.841
286,NOL11,14,10,autoencoder_cosine; combined_pearson_jaccard; combined_s...,11.957
476,WDR3,14,10,autoencoder_cosine; combined_pearson_jaccard; combined_s...,11.957
470,UTP6,13,10,autoencoder_cosine; combined_pearson_jaccard; combined_s...,11.889
139,HEATR1,15,10,autoencoder_cosine; combined_pearson_jaccard; combined_s...,11.951
342,PSMC2,11,9,autoencoder_cosine; combined_pearson_jaccard; combined_s...,10.948
467,UTP15,15,9,autoencoder_cosine; combined_pearson_jaccard; combined_s...,11.721
354,PSMD6,9,9,autoencoder_cosine; combined_pearson_jaccard; combined_s...,11.037
353,PSMD3,9,9,autoencoder_cosine; combined_pearson_jaccard; combined_s...,11.037


### Table 8 — MCODE Core Size & Density Distribution

Distribution of all detected MCODE cores per (method, strategy). Helps assess whether current parameters are too loose or too strict.

In [9]:
# Load all per-run mcode_cores.csv files and concatenate.
all_cores = []

for method in sorted(os.listdir(ROOT)):
    method_path = os.path.join(ROOT, method)
    if not os.path.isdir(method_path):
        continue
    for strategy in sorted(os.listdir(method_path)):
        cores_f = os.path.join(method_path, strategy, "mcode_cores.csv")
        if os.path.exists(cores_f):
            all_cores.append(pd.read_csv(cores_f))

all_cores_df = pd.concat(all_cores, ignore_index=True)

# Distribution stats per (method, strategy)
t8 = (
    all_cores_df
    .groupby(["method", "strategy"])
    .agg(
        n_cores=("core_id", "count"),
        size_min=("size", "min"),
        size_q25=("size", lambda x: x.quantile(0.25)),
        size_median=("size", "median"),
        size_q75=("size", lambda x: x.quantile(0.75)),
        size_max=("size", "max"),
        density_median=("density", "median"),
        density_q25=("density", lambda x: x.quantile(0.25)),
        score_median=("mcode_score", "median"),
        score_max=("mcode_score", "max"),
        # fraction of cores considered "complex-like": density ≥ 0.7 and size ≤ 50
        n_complex_like=("size", lambda x: ((x <= 50) & (all_cores_df.loc[x.index, "density"] >= 0.7)).sum()),
    )
    .round(2)
    .reset_index()
    .sort_values(["method", "strategy"])
)

t8["pct_complex_like"] = (t8["n_complex_like"] / t8["n_cores"] * 100).round(1)

display(t8)

,method,strategy,n_cores,size_min,size_q25,size_median,size_q75,size_max,density_median,density_q25,score_median,score_max,n_complex_like,pct_complex_like
0,autoencoder_cosine,threshold_0p85,1051,5,8.000,11.000,19.000,99,0.820,0.690,8.500,94.730,751,71.500
1,autoencoder_cosine,top100000,1393,5,8.000,12.000,22.000,106,0.790,0.670,9.000,102.570,911,65.400
2,autoencoder_cosine,top10perNodes,976,5,7.000,9.000,10.000,22,0.710,0.580,5.780,12.290,498,51.000
3,combined_pearson_jaccard,threshold_0p85,1235,5,9.000,15.000,28.000,149,0.860,0.760,12.130,133.070,989,80.100
4,combined_pearson_jaccard,top100000,1235,5,9.000,15.000,28.000,149,0.860,0.760,12.130,133.070,989,80.100
5,combined_pearson_jaccard,top10perNodes,935,5,7.000,9.000,10.000,23,0.710,0.600,5.780,13.450,511,54.700
6,combined_spearman_jaccard,threshold_0p85,1153,5,9.000,15.000,27.000,185,0.850,0.750,12.000,158.320,856,74.200
7,combined_spearman_jaccard,top100000,1153,5,9.000,15.000,27.000,185,0.850,0.750,12.000,158.320,856,74.200
8,combined_spearman_jaccard,top10perNodes,839,5,7.000,9.000,10.000,18,0.730,0.610,6.000,12.710,482,57.400
9,composite_rbf_jaccard,threshold_0p85,746,5,9.000,19.000,42.000,249,0.900,0.820,16.380,205.710,558,74.800
